# HW2 Benchmark

This notebook compares a naive regression baseline with a simple ML benchmark for pre-run reliability prediction on the generated `thesis_production_125k_v1` dataset sample saved in `data/hw2/`. The Kaggle NISQ Fault Logs 100K dataset is historical background for this repository, but it is not the dataset evaluated in this HW2 benchmark.

The comparison uses the validation split only. The test split is loaded only to confirm that it exists and remains reserved for later final evaluation.

In [1]:
import importlib.util
import subprocess
import sys

_REQUIRED_IMPORTS = [
    ('pandas', 'pandas'),
    ('pyarrow', 'pyarrow'),
    ('sklearn', 'scikit-learn'),
]

_missing_pip_packages = []
for import_name, pip_name in _REQUIRED_IMPORTS:
    if importlib.util.find_spec(import_name) is None:
        _missing_pip_packages.append(pip_name)

if _missing_pip_packages:
    _cmd = [sys.executable, '-m', 'pip', 'install', *_missing_pip_packages]
    print('Installing missing notebook dependencies with:')
    print(' '.join(_cmd))
    subprocess.check_call(_cmd)
else:
    print('Notebook dependencies are available.')


Notebook dependencies are available.


## 1. Task And Metric

Task: predict continuous `reliability` from pre-run circuit and hardware features.

Primary validation metric: MAE, because it is directly interpretable as average absolute reliability error on a `[0, 1]` target. RMSE and R2 are reported as supporting metrics.


In [2]:
from pathlib import Path
import json

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
HW2_DATA_DIR = Path('data/hw2')
train = pd.read_parquet(HW2_DATA_DIR / 'train.parquet')
validation = pd.read_parquet(HW2_DATA_DIR / 'validation.parquet')
test = pd.read_parquet(HW2_DATA_DIR / 'test.parquet')
feature_policy = json.loads((HW2_DATA_DIR / 'feature_policy.json').read_text(encoding='utf-8'))

print(f'Train rows: {len(train):,}')
print(f'Validation rows: {len(validation):,}')
print(f'Test rows, reserved and not evaluated here: {len(test):,}')


Train rows: 20,000
Validation rows: 2,500
Test rows, reserved and not evaluated here: 2,500


## 2. Train-Only Preprocessing Pipeline

The imputer, encoder, and scaler are inside sklearn pipelines. They are fitted only on the training split. Validation rows are transformed with train-derived parameters.


In [3]:
target_column = feature_policy['target_column']
feature_columns = feature_policy['allowed_feature_columns']

X_train = train[feature_columns]
y_train = train[target_column].astype(float)
X_validation = validation[feature_columns]
y_validation = validation[target_column].astype(float)

categorical_columns = X_train.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_columns),
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_columns),
    ],
    remainder='drop',
)

print(f'Categorical columns ({len(categorical_columns)}):', categorical_columns)
print(f'Numeric columns ({len(numeric_columns)}):', numeric_columns[:12], '...')


Categorical columns (0): []
Numeric columns (46): ['qubit_count', 'original_circuit_depth', 'original_num_cx', 'original_num_cz', 'original_num_single_qubit_gates', 'original_entanglement_density', 'original_unique_two_qubit_edge_count', 'original_interaction_graph_density', 'original_mean_qubit_gate_count', 'original_max_qubit_gate_count', 'original_mean_qubit_two_qubit_gate_count', 'original_max_qubit_two_qubit_gate_count'] ...


## 3. Naive Baseline And Simple ML Benchmark

The naive baseline predicts the training-set mean reliability. Ridge regression is a simple linear model with regularization, which is appropriate for HW2 because it tests whether a transparent baseline can use the pre-run features without moving into advanced model tuning.


In [4]:
def regression_metrics(y_true, y_pred):
    return {
        'validation_mae': mean_absolute_error(y_true, y_pred),
        'validation_rmse': mean_squared_error(y_true, y_pred) ** 0.5,
        'validation_r2': r2_score(y_true, y_pred),
    }

models = {
    'dummy_mean': DummyRegressor(strategy='mean'),
    'ridge_regression': Ridge(alpha=1.0, random_state=RANDOM_STATE),
}

results = []
for model_name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_validation)
    row = {'model': model_name}
    row.update(regression_metrics(y_validation, predictions))
    results.append(row)

comparison = pd.DataFrame(results).sort_values('validation_mae', ascending=True).reset_index(drop=True)
comparison


,model,validation_mae,validation_rmse,validation_r2
0,ridge_regression,0.072702,0.109432,0.664128
1,dummy_mean,0.145326,0.188831,-0.000074


## 4. Validation-Only Evaluation

The table above is the HW2 benchmark comparison. The test set is intentionally not evaluated here, because it should remain untouched until the final model/evaluation stage.


In [5]:
print('Validation target summary:')
print(y_validation.describe().to_string())
print()
print('Benchmark comparison:')
print(comparison.to_string(index=False))


Validation target summary:
count    2500.000000
mean        0.791551
std         0.188862
min         0.010742
25%         0.726562
50%         0.848145
75%         0.925781
max         1.000000

Benchmark comparison:
           model  validation_mae  validation_rmse  validation_r2
ridge_regression        0.072702         0.109432       0.664128
      dummy_mean        0.145326         0.188831      -0.000074


## 5. Short Summary

The reliability task is a regression problem, not fault-type classification. The benchmark confirms that the project now has a clean reference point: a train-mean naive baseline and a simple Ridge model, both evaluated only on validation data with train-only preprocessing. The next modeling step can compare stronger regressors while keeping the grouped test set sealed.
